# Proyek Machine Learning: Prediksi Stroke

## Bagian A: Pemodelan di Google Colab

### 1. Definisi Masalah & Pemuatan Data

In [ ]:
# Impor library yang diperlukan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Impor kagglehub untuk mengakses dataset
import kagglehub

# Download dataset
path = kagglehub.dataset_download("fedesoriano/stroke-prediction-dataset")
print("Path to dataset files:", path)

# Muat dataset dari path yang diunduh
import os
dataset_files = os.listdir(path)
print("File dalam dataset:", dataset_files)

# Ambil nama file CSV pertama (biasanya hanya ada satu)
csv_file = [f for f in dataset_files if f.endswith('.csv')][0]
file_path = os.path.join(path, csv_file)

# Muat dataset
df = pd.read_csv(file_path)

# Tampilkan informasi dasar tentang dataset
print("\nBentuk Dataset:", df.shape)
print("\nInfo Dataset:")
print(df.info())
print("\nBeberapa baris pertama:")
df.head()

In [ ]:
# Periksa statistik dasar
df.describe()

### 2. Eksplorasi Data (EDA)

In [ ]:
# Periksa distribusi target
plt.figure(figsize=(8, 5))
sns.countplot(x='stroke', data=df)
plt.title('Distribusi Stroke')
plt.show()

print("Distribusi Stroke:")
print(df['stroke'].value_counts())
print(f"Tingkat Stroke: {df['stroke'].mean():.3f}")

In [ ]:
# Distribusi fitur numerik berdasarkan stroke
numerical_features = ['age', 'avg_glucose_level', 'bmi']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feature in enumerate(numerical_features):
    sns.histplot(data=df, x=feature, hue='stroke', ax=axes[i], kde=True)
    axes[i].set_title(f'Distribusi {feature} berdasarkan Stroke')
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur kategorik berdasarkan stroke
categorical_features = ['gender', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for i, feature in enumerate(categorical_features):
    sns.countplot(data=df, x=feature, hue='stroke', ax=axes[i])
    axes[i].set_title(f'{feature} vs Stroke')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Matriks korelasi untuk fitur numerik
plt.figure(figsize=(10, 6))
correlation_matrix = df[['age', 'avg_glucose_level', 'bmi', 'stroke']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Matriks Korelasi Fitur Numerik')
plt.show()

### 3. Persiapan & Pra-pemrosesan Data

In [ ]:
# Periksa nilai yang hilang
print("Nilai yang hilang di setiap kolom:")
print(df.isnull().sum())

In [ ]:
# Tangani nilai yang hilang (terutama pada kolom BMI)
# Untuk kolom numerik, kita gunakan median
# Untuk kolom kategorik, kita gunakan mode

# Tangani nilai BMI yang hilang dengan median
df['bmi'].fillna(df['bmi'].median(), inplace=True)

# Hapus baris dengan gender 'Other' karena jumlahnya sangat sedikit
df = df[df['gender'] != 'Other']

print("Nilai yang hilang setelah preprocessing:")
print(df.isnull().sum())

In [ ]:
# Pisahkan fitur dan target
X = df.drop('stroke', axis=1)
y = df['stroke']

In [ ]:
# Identifikasi kolom kategorik dan numerik
categorical_columns = X.select_dtypes(include=['object']).columns.tolist()
numerical_columns = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print("Kolom kategorik:", categorical_columns)
print("Kolom numerik:", numerical_columns)

In [ ]:
# Encode variabel kategorik
label_encoders = {}
X_encoded = X.copy()

for col in categorical_columns:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print("Bentuk dataset ter-encode:", X_encoded.shape)
X_encoded.head()

In [ ]:
# Skala fitur numerik
scaler = StandardScaler()
X_scaled = X_encoded.copy()
X_scaled[numerical_columns] = scaler.fit_transform(X_encoded[numerical_columns])

print("Bentuk dataset ter-skalakan:", X_scaled.shape)
X_scaled.head()

### 4. Pelatihan Model

In [ ]:
# Bagi dataset menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"Ukuran data latih: {X_train.shape[0]}")
print(f"Ukuran data uji: {X_test.shape[0]}")
print(f"Tingkat stroke di data latih: {y_train.mean():.3f}")
print(f"Tingkat stroke di data uji: {y_test.mean():.3f}")

In [ ]:
# Inisialisasi model
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', random_state=42)
}

In [ ]:
# Latih dan evaluasi model
model_results = {}

for name, model in models.items():
    print(f"\nMelatih {name}...")
    
    # Latih model
    model.fit(X_train, y_train)
    
    # Buat prediksi
    y_pred = model.predict(X_test)
    
    # Hitung metrik
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    model_results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'predictions': y_pred
    }
    
    print(f"Hasil {name}:")
    print(f"  Akurasi: {accuracy:.4f}")
    print(f" Presisi: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")

### 5. Evaluasi Model

In [ ]:
# Buat dataframe perbandingan
comparison_df = pd.DataFrame({
    'Model': list(model_results.keys()),
    'Akurasi': [model_results[name]['accuracy'] for name in model_results.keys()],
    'Presisi': [model_results[name]['precision'] for name in model_results.keys()],
    'Recall': [model_results[name]['recall'] for name in model_results.keys()],
    'F1-Score': [model_results[name]['f1_score'] for name in model_results.keys()]
})

print("Perbandingan Model:")
print(comparison_df)

# Visualisasikan kinerja model
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

metrics = ['Akurasi', 'Presisi', 'Recall', 'F1-Score']
for i, metric in enumerate(metrics):
    ax = axes[i//2, i%2]
    ax.bar(comparison_df['Model'], comparison_df[metric])
    ax.set_title(f'Perbandingan {metric}')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Temukan model terbaik berdasarkan F1-score (yang menyeimbangkan presisi dan recall)
best_model_name = max(model_results, key=lambda x: model_results[x]['f1_score'])
best_model = model_results[best_model_name]['model']

print(f"\nModel Terbaik: {best_model_name}")
print(f"F1-Score: {model_results[best_model_name]['f1_score']:.4f}")
print(f"Akurasi: {model_results[best_model_name]['accuracy']:.4f}")
print(f"Presisi: {model_results[best_model_name]['precision']:.4f}")
print(f"Recall: {model_results[best_model_name]['recall']:.4f}")

# Laporan klasifikasi terperinci untuk model terbaik
print(f"\nLaporan Klasifikasi Terperinci untuk {best_model_name}:")
print(classification_report(y_test, model_results[best_model_name]['predictions']))

In [ ]:
# Matriks Konfusi untuk model terbaik
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, model_results[best_model_name]['predictions'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Matriks Konfusi - {best_model_name}')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.show()

### 6. Simpan Model

In [ ]:
# Impor pickle untuk menyimpan model dan objek pra-pemrosesan
import pickle

# Simpan model terbaik, scaler, dan encoder label
model_data = {
    'model': best_model,
    'scaler': scaler,
    'label_encoders': label_encoders,
    'numerical_columns': numerical_columns,
    'categorical_columns': categorical_columns
}

# Simpan ke file
with open('model_terbaik.pkl', 'wb') as file:
    pickle.dump(model_data, file)

print("Model dan objek pra-pemrosesan disimpan ke 'model_terbaik.pkl'")

# Simpan komponen-komponen secara terpisah
# Simpan model terbaik saja
with open('model terbaik.pkl', 'wb') as file:
    pickle.dump(best_model, file)

# Simpan scaler
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

# Simpan encoders
with open('encoders.pkl', 'wb') as file:
    pickle.dump(label_encoders, file)

print("\nKomponen-komponen model juga disimpan secara terpisah:")
print("- model terbaik.pkl (model terbaik saja)")
print("- scaler.pkl (StandardScaler)")
print("- encoders.pkl (LabelEncoders)")

# Verifikasi file dibuat
import os
if os.path.exists('model_terbaik.pkl'):
    print("\nFile 'model_terbaik.pkl' sudah ada dan siap digunakan.")
else:
    print("\nError: File 'model_terbaik.pkl' tidak dibuat.")

## Ringkasan

Di notebook ini, kami telah menyelesaikan langkah-langkah berikut:
1. **Definisi Masalah**: Memprediksi kejadian stroke menggunakan dataset stroke asli dari Kaggle
2. **Eksplorasi Data**: Menganalisis distribusi fitur dan korelasi
3. **Pra-pemrosesan Data**: Menangani nilai yang hilang, menangani variabel kategorik dengan encoding label, dan menskalakan fitur numerik
4. **Pelatihan Model**: Melatih 4 model berbeda (Logistic Regression, Random Forest, KNN, SVM)
5. **Evaluasi Model**: Membandingkan model menggunakan akurasi, presisi, recall, dan F1-score
6. **Penyimpanan Model**: Menyimpan model terbaik beserta objek pra-pemrosesan

Model terbaik dipilih berdasarkan F1-Score tertinggi, yang menyeimbangkan presisi dan recall, menjadikannya model pilihan untuk deployment.